In [0]:
%sql
---- Creating new catalog, schema -----
use catalog sql_youtube_practise;
create schema if not exists pyspark;
use pyspark;
show current schema;

catalog,namespace
sql_youtube_practise,pyspark


##### Question1: Business city table has data from the day udaan has started operation. 
Write a SQL to identify year-wise count of new cities where udaan started their operations.

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
business_city_data = [
    ("2020-01-02", 3),
    ("2020-07-01", 7),
    ("2021-01-01", 3),
    ("2021-02-03", 19),
    ("2022-12-01", 3),
    ("2022-12-15", 3),
    ("2022-02-28", 12)
]
business_city_schema = StructType([
    StructField("business_date", StringType(), True),
    StructField("city_id", IntegerType(), True)
])
business_city_df = spark.createDataFrame(
    business_city_data,
    schema=business_city_schema
)
# Convert string to date
business_city_df = business_city_df.withColumn(
    "business_date",
    to_date("business_date", "yyyy-MM-dd")
)
business_city_df.write.mode("overwrite").saveAsTable("business_city")
business_city_df.orderBy("business_date").display()

business_date,city_id
2020-01-02,3
2020-07-01,7
2021-01-01,3
2021-02-03,19
2022-02-28,12
2022-12-01,3
2022-12-15,3


In [0]:
from pyspark.sql.functions import *
df = business_city_df.groupBy(
    col("city_id")
).agg(
    min(col("business_date")).alias("date")
).withColumn(
    "year",
    year(col("date"))
)

df1 = df.groupBy(
    col("year")
).agg(
    countDistinct(col("city_id")).alias("count")
)

df.display()
df1.display()

city_id,date,year
3,2020-01-02,2020
7,2020-07-01,2020
19,2021-02-03,2021
12,2022-02-28,2022


year,count
2020,2
2021,1
2022,1


##### Question 4: determine phone numbers that satisfy below conditions: 
> - the numbers have both incoming & outgoing calls
> - the sum of duration of outgoing calls should be greater than sum of duration of incoming calls

In [0]:
from pyspark.sql.types import *
call_details_data = [
    ("OUT", "181868", 13),
    ("OUT", "2159010", 8),
    ("OUT", "2159010", 178),
    ("SMS", "4153810", 1),
    ("OUT", "2159010", 152),
    ("OUT", "9140152", 18),
    ("SMS", "4162672", 1),
    ("SMS", "9168204", 1),
    ("OUT", "9168204", 576),
    ("INC", "2159010", 5),
    ("INC", "2159010", 4),
    ("SMS", "2159010", 1),
    ("SMS", "4535614", 1),
    ("OUT", "181868", 20),
    ("INC", "181868", 54),
    ("INC", "218748", 20),
    ("INC", "2159010", 9),
    ("INC", "197432", 66),
    ("SMS", "2159010", 1),
    ("SMS", "4535614", 1)
]
call_details_schema = StructType([
    StructField("call_type", StringType(), True),
    StructField("call_number", StringType(), True),
    StructField("call_duration", IntegerType(), True)
])
call_details_df = spark.createDataFrame(
    call_details_data,
    schema=call_details_schema
)
call_details_df.write.mode("Overwrite").saveAsTable("call_details")
call_details_df.display()

call_type,call_number,call_duration
OUT,181868,13
OUT,2159010,8
OUT,2159010,178
SMS,4153810,1
OUT,2159010,152
OUT,9140152,18
SMS,4162672,1
SMS,9168204,1
OUT,9168204,576
INC,2159010,5


In [0]:

from pyspark.sql.functions import *

df = call_details_df.groupBy(
    col("call_number")
).agg(
    sum(
        when (
            col("call_type") == "OUT",
            col("call_duration")
        )
    ).alias("out_duration"),
    sum(
        when (
            col("call_type") == "INC",
            col("call_duration")
        )
    ).alias("inc_duration")
).filter(
    col("out_duration").isNotNull() & col("inc_duration").isNotNull()
).filter(
    col("out_duration") > col("inc_duration")
)

df.display()

call_number,out_duration,inc_duration
2159010,338,18


##### Question6: populate category values to the last not null values

In [0]:
from pyspark.sql.types import *
brands_data = [
    ("chocolates", "5-star"),
    (None, "dairy milk"),
    (None, "perk"),
    (None, "eclair"),
    ("Biscuits", "britannia"),
    (None, "good day"),
    (None, "boost")
]
brands_schema = StructType([
    StructField("category", StringType(), True),
    StructField("brand_name", StringType(), True)
])
brands_df = spark.createDataFrame(
    brands_data,
    schema=brands_schema
)

brands_df.write.mode("Overwrite").saveAsTable("brands")
brands_df.display()

category,brand_name
chocolates,5-star
null,dairy milk
null,perk
null,eclair
Biscuits,britannia
null,good day
null,boost


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

df = brands_df.withColumn(
    "rn",
    row_number().over(Window.orderBy(lit(1)))
)

df1 = df.filter(
    col("category").isNotNull()
).withColumn(
    "next_rn",
    lead("rn", 1, 9999).over(
        Window.orderBy("rn")
    )
)

final_df = df.alias("df").join(
    df1.alias("df1"),
    (col("df.rn") >= col("df1.rn")) & (col("df.rn") < col("df1.next_rn")),
    "inner"
).select(
    col("df1.category"),
    col("df.brand_name")
)

df.display()
df1.display()
final_df.display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


category,brand_name,rn
chocolates,5-star,1
null,dairy milk,2
null,perk,3
null,eclair,4
Biscuits,britannia,5
null,good day,6
null,boost,7


category,brand_name,rn,next_rn
chocolates,5-star,1,5
Biscuits,britannia,5,9999


category,brand_name
chocolates,5-star
chocolates,perk
Biscuits,good day
chocolates,dairy milk
chocolates,eclair
Biscuits,britannia
Biscuits,boost


#### Question7: SQL query to report the students (student_id, student_name) being "quiet" in all exams. 
- "quiet" student is the one who took at least one exam and didn't score neither the high score nor the low score in any. 
- Don't return the student who has never taken any exams. 
- Return the result table ordered by student id.

In [0]:
from pyspark.sql.types import *
# --------------------
# Students
# --------------------
students_data = [ (1, "Daniel"), (2, "Jade"), (3, "Stella"), (4, "Jonathan"), (5, "Will") ]
students_schema = StructType([
    StructField("student_id", IntegerType(), True),
    StructField("student_name", StringType(), True)
])
students_df = spark.createDataFrame(
    students_data,
    schema=students_schema
)
students_df.write.mode("Overwrite").saveAsTable("students1")
students_df.display()

# --------------------
# Exams
# --------------------
exams_data = [ (10, 1, 70), (10, 2, 80), (10, 3, 90), (20, 1, 80), (30, 1, 70), (30, 3, 80), (30, 4, 90), (40, 1, 60), (40, 2, 70), (40, 4, 80) ]
exams_schema = StructType([
    StructField("exam_id", IntegerType(), True),
    StructField("student_id", IntegerType(), True),
    StructField("score", IntegerType(), True)
])
exams_df = spark.createDataFrame(
    exams_data,
    schema=exams_schema
)
exams_df.write.mode("Overwrite").saveAsTable("exams1")
exams_df.display()

student_id,student_name
1,Daniel
2,Jade
3,Stella
4,Jonathan
5,Will


exam_id,student_id,score
10,1,70
10,2,80
10,3,90
20,1,80
30,1,70
30,3,80
30,4,90
40,1,60
40,2,70
40,4,80


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

df = exams_df.withColumn(
    "max_score",
    max(col("score")).over(
        Window.partitionBy(col("exam_id"))
    )
).withColumn(
    "min_score",
    min(col("score")).over(
        Window.partitionBy(col("exam_id"))
    )
)

final_df = df.alias("a").join(
    students_df.alias("s"),
    col("a.student_id") == col("s.student_id")
).filter(
    (col("score") > col("min_score")) &
    (col("score") < col("max_score"))
).groupBy(
    col("a.student_id"),
    col("s.student_name")
).agg(
    count(col("a.student_id")).alias("count")
)

total_exams = exams_df.select("exam_id").distinct().count()

final_df1 = final_df.filter(
    col("count") == total_exams
).select(
    "student_id",
    "student_name"
).orderBy(
    "student_id"
)

df.display()
final_df.display()
final_df1.display()

exam_id,student_id,score,max_score,min_score
10,1,70,90,70
10,2,80,90,70
10,3,90,90,70
20,1,80,80,80
30,1,70,90,70
30,3,80,90,70
30,4,90,90,70
40,1,60,80,60
40,2,70,80,60
40,4,80,80,60


student_id,student_name,count
2,Jade,2
3,Stella,1


student_id,student_name
